# BDA Blueprint Instruction Optimization - Purchase Order Workshop

This notebook walks through the BDA Blueprint Instruction Optimization API using Acme Bikes purchase order documents.

1. Initialize clients and S3 bucket
2. Upload sample purchase order document to S3
3. Create a blueprint from schema
4. Review ground truth data
5. Run optimization with 3 document samples
6. Retrieve and compare optimized blueprint

**Prerequisites:** AWS account with BDA access, configured credentials, Python 3.8+

In [1]:
import boto3
import json
import time
import os
from IPython.display import display, IFrame

print(f"boto3 version: {boto3.__version__}")

boto3 version: 1.42.48


## 1. Initialize Clients and S3 Bucket

In [2]:
session = boto3.Session()
region = session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

bda_client = boto3.client('bedrock-data-automation', region_name=region)
s3_client = boto3.client('s3', region_name=region)

bucket_name = os.environ['BDA_BUCKET']
profile_arn = f"arn:aws:bedrock:{region}:{account_id}:data-automation-profile/us.data-automation-v1"

print(f"Region: {region}")
print(f"Account: {account_id}")
print(f"Bucket: {bucket_name}")
print(f"Profile ARN: {profile_arn}")

Using existing bucket: bda-workshop-us-east-1-208264011282
Region: us-east-1
Account: 208264011282
Profile ARN: arn:aws:bedrock:us-east-1:208264011282:data-automation-profile/us.data-automation-v1


## 2. Upload Sample Purchase Order to S3

In [3]:
local_pdf_path = "sample_data/purchase_order/PO_RT-003_Mountain_Gear_Co.pdf"

s3_key = "bda/input/purchase_order/PO_RT-003_Mountain_Gear_Co.pdf"
s3_client.upload_file(local_pdf_path, bucket_name, s3_key)
document_s3_uri = f"s3://{bucket_name}/{s3_key}"
print(f"Uploaded to: {document_s3_uri}")

Uploaded to: s3://bda-workshop-us-east-1-208264011282/bda/input/purchase_order/PO_RT-003_Mountain_Gear_Co.pdf


In [4]:
display(IFrame(local_pdf_path, width=800, height=500))

## 3. Create Blueprint with `CreateBlueprint`

Loading the schema from `sample_data/purchase_order/bda_schema.json`.

In [5]:
with open('sample_data/purchase_order/bda_schema.json') as f:
    blueprint_schema = json.load(f)

# Save original instructions for later comparison
original_instructions = {
    name: prop["instruction"]
    for name, prop in blueprint_schema["properties"].items()
}

print("Blueprint schema:")
print(json.dumps(blueprint_schema, indent=2))

Blueprint schema:
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "description": "A purchase order document from Cycle City to a customer",
  "class": "Purchase Order",
  "type": "object",
  "definitions": {
    "OrderItem": {
      "type": "object",
      "properties": {
        "sku": {
          "type": "string",
          "inferenceType": "explicit",
          "instruction": "The stock keeping unit of the item"
        },
        "product_name": {
          "type": "string",
          "inferenceType": "explicit",
          "instruction": "The name of the product"
        },
        "description": {
          "type": "string",
          "inferenceType": "explicit",
          "instruction": "A detailed description of the product"
        },
        "quantity": {
          "type": "number",
          "inferenceType": "explicit",
          "instruction": "The quantity of the product ordered"
        },
        "unit_price": {
          "type": "number",
          "inference

In [6]:
blueprint_name = f"acme-bikes-purchase-order-{int(time.time())}"

response = bda_client.create_blueprint(
    blueprintName=blueprint_name,
    type='DOCUMENT',
    blueprintStage='DEVELOPMENT',
    schema=json.dumps(blueprint_schema)
)

blueprint_arn = response['blueprint']['blueprintArn']
print(f"Blueprint created: {blueprint_name}")
print(f"ARN: {blueprint_arn}")

Blueprint created: acme-bikes-purchase-order-1771965002
ARN: arn:aws:bedrock:us-east-1:208264011282:blueprint/77ff645c8614


## 4. Review Ground Truth Data

In [7]:
gt_path = "sample_data/purchase_order/PO_RT-003_Mountain_Gear_Co_ground_truth.json"

with open(gt_path) as f:
    ground_truth = json.load(f)

print("Ground truth for Mountain Gear Co PO:")
print(json.dumps(ground_truth, indent=2))

Ground truth for Mountain Gear Co PO:
{
  "po_number": "PO-2026-0224-0524",
  "retailer_name": "Mountain Gear Co",
  "order_date": "2026-02-24",
  "order_total": 9745.53,
  "customer_name": "John Hall",
  "special_requests": "",
  "order_items": [
    {
      "sku": "AB-EB-017",
      "product_name": "E-Ride Master",
      "description": "E-Ride Master",
      "quantity": 5,
      "unit_price": 1860.69,
      "line_total": 9303.45,
      "option_color": "cb8901",
      "option_size": "S",
      "option_weight": "19.4 lbs",
      "option_battery": "48V 17.5Ah",
      "option_range": "41 miles",
      "option_speed": "",
      "option_wheel_size": "",
      "option_brake_type": "",
      "option_certification": ""
    },
    {
      "sku": "AB-KB-033",
      "product_name": "Small Classic",
      "description": "Small Classic",
      "quantity": 2,
      "unit_price": 221.04,
      "line_total": 442.08,
      "option_color": "ol4567",
      "option_size": "",
      "option_weight": "",
 

## 5. Run Blueprint Optimization with `InvokeBlueprintOptimizationAsync`

Using 3 purchase order samples for optimization.

In [8]:
# Upload all 3 sample sets to S3
samples_config = [
    {
        'pdf': 'sample_data/purchase_order/PO_RT-002_Bike_World.pdf',
        'gt':  'sample_data/purchase_order/PO_RT-002_Bike_World_ground_truth.json',
        's3_prefix': 'bda/input/purchase_order/PO_RT-002_Bike_World'
    },
    {
        'pdf': 'sample_data/purchase_order/PO_RT-003_Mountain_Gear_Co.pdf',
        'gt':  'sample_data/purchase_order/PO_RT-003_Mountain_Gear_Co_ground_truth.json',
        's3_prefix': 'bda/input/purchase_order/PO_RT-003_Mountain_Gear_Co'
    },
    {
        'pdf': 'sample_data/purchase_order/PO_RT-004_Urban_Cycles.pdf',
        'gt':  'sample_data/purchase_order/PO_RT-004_Urban_Cycles_ground_truth.json',
        's3_prefix': 'bda/input/purchase_order/PO_RT-004_Urban_Cycles'
    }
]

samples = []
for s in samples_config:
    pdf_key = f"{s['s3_prefix']}.pdf"
    gt_key  = f"{s['s3_prefix']}_ground_truth.json"
    s3_client.upload_file(s['pdf'], bucket_name, pdf_key)
    s3_client.upload_file(s['gt'],  bucket_name, gt_key)
    samples.append({
        'assetS3Object':       {'s3Uri': f"s3://{bucket_name}/{pdf_key}"},
        'groundTruthS3Object': {'s3Uri': f"s3://{bucket_name}/{gt_key}"}
    })
    print(f"Uploaded: {s['s3_prefix']}")

print(f"\n{len(samples)} sample sets ready")

Uploaded: bda/input/purchase_order/PO_RT-002_Bike_World
Uploaded: bda/input/purchase_order/PO_RT-003_Mountain_Gear_Co
Uploaded: bda/input/purchase_order/PO_RT-004_Urban_Cycles

3 sample sets ready


In [9]:
optimization_output_uri = f"s3://{bucket_name}/bda/optimization-output/purchase-order/"

response = bda_client.invoke_blueprint_optimization_async(
    blueprint={
        'blueprintArn': blueprint_arn,
        'stage': 'DEVELOPMENT'
    },
    samples=samples,
    outputConfiguration={
        's3Object': {'s3Uri': optimization_output_uri}
    },
    dataAutomationProfileArn=profile_arn
)

invocation_arn = response['invocationArn']
print(f"Optimization job started")
print(f"Invocation ARN: {invocation_arn}")

Optimization job started
Invocation ARN: arn:aws:bedrock:us-east-1:208264011282:blueprint-optimization-invocation/d9d0b1f2-a104-4472-b368-002d3dac4829


## 6. Poll Status with `GetBlueprintOptimizationStatus`

In [10]:
print("Polling optimization status...\n")

while True:
    status_response = bda_client.get_blueprint_optimization_status(
        invocationArn=invocation_arn
    )
    status = status_response.get('status', 'Unknown')
    print(f"  Status: {status}")

    if status == 'Success':
        print(f"\nOptimization completed.")
        break
    elif status in ('ServiceError', 'ClientError'):
        print(f"\nOptimization failed: {status_response.get('errorMessage', 'N/A')}")
        break

    time.sleep(15)

Polling optimization status...

  Status: InProgress
  Status: InProgress
  Status: InProgress
  Status: InProgress
  Status: InProgress
  Status: InProgress
  Status: Success

Optimization completed.


## 7. Retrieve Optimized Blueprint with `GetBlueprint`

In [11]:
bp_response = bda_client.get_blueprint(
    blueprintArn=blueprint_arn,
    blueprintStage='DEVELOPMENT'
)

optimized_schema = json.loads(bp_response['blueprint']['schema'])

print("Optimized schema:")
print(json.dumps(optimized_schema, indent=2))

Optimized schema:
{
  "class": "Purchase Order",
  "description": "A purchase order document from Cycle City to a customer",
  "definitions": {
    "OrderItem": {
      "properties": {
        "sku": {
          "instruction": "The stock keeping unit of the item",
          "inferenceType": "explicit",
          "type": "string"
        },
        "product_name": {
          "instruction": "The name of the product",
          "inferenceType": "explicit",
          "type": "string"
        },
        "description": {
          "instruction": "The product name or model description in the order items table, typically appearing after product codes (like AB-XX-XXX) and before specifications such as colors, sizes, or weights.",
          "inferenceType": "explicit",
          "type": "string"
        },
        "quantity": {
          "instruction": "The quantity of the product ordered",
          "inferenceType": "explicit",
          "type": "number"
        },
        "unit_price": {
    

In [12]:
def compare_instructions(original_props, optimized_props, definitions=None, indent=0):
    prefix = '  ' * indent
    for field_name, orig_prop in original_props.items():
        # Resolve $ref
        if '$ref' in orig_prop and definitions:
            ref_name = orig_prop['$ref'].split('/')[-1]
            orig_prop = definitions.get(ref_name, orig_prop)

        opt_prop = optimized_props.get(field_name, {})

        if 'instruction' in orig_prop:
            orig_instr = orig_prop['instruction']
            opt_instr = opt_prop.get('instruction', 'N/A')
            changed = orig_instr != opt_instr
            print(f"\n{prefix}{field_name} {'[CHANGED]' if changed else '[UNCHANGED]'}")
            print(f"{prefix}  Before: {orig_instr}")
            print(f"{prefix}  After:  {opt_instr}")

        # Recurse into array items
        if orig_prop.get('type') == 'array' and 'items' in orig_prop:
            items_def = orig_prop['items']
            if '$ref' in items_def and definitions:
                ref_name = items_def['$ref'].split('/')[-1]
                items_def = definitions.get(ref_name, {})
            if 'properties' in items_def:
                opt_items = opt_prop.get('items', {})
                if '$ref' in opt_prop.get('items', {}) and definitions:
                    ref_name = opt_prop['items']['$ref'].split('/')[-1]
                    opt_items = definitions.get(ref_name, {})
                print(f"{prefix}  [{field_name} items]")
                compare_instructions(
                    items_def['properties'],
                    opt_items.get('properties', {}),
                    definitions,
                    indent + 2
                )

        # Recurse into nested objects
        elif orig_prop.get('type') == 'object' and 'properties' in orig_prop:
            compare_instructions(
                orig_prop['properties'],
                opt_prop.get('properties', {}),
                definitions,
                indent + 1
            )

print("Instruction comparison (before -> after optimization):")
print("=" * 80)

orig_defs = blueprint_schema.get('definitions', {})
opt_defs  = optimized_schema.get('definitions', {})

# Merge definitions so nested lookups work for both original and optimized
merged_defs = {**orig_defs, **{k: opt_defs[k] for k in opt_defs}}

compare_instructions(
    blueprint_schema['properties'],
    optimized_schema.get('properties', {}),
    merged_defs
)

print("\n" + "=" * 80)

Instruction comparison (before -> after optimization):

po_number [UNCHANGED]
  Before: The unique identifier for the purchase order
  After:  The unique identifier for the purchase order

retailer_account [UNCHANGED]
  Before: The account number of the retailer
  After:  The account number of the retailer

retailer_name [UNCHANGED]
  Before: The name of the retailer
  After:  The name of the retailer

order_date [UNCHANGED]
  Before: The date when the order was placed
  After:  The date when the order was placed

order_total [UNCHANGED]
  Before: The total amount for the order
  After:  The total amount for the order

customer_name [UNCHANGED]
  Before: The name of the customer
  After:  The name of the customer

customer_email [UNCHANGED]
  Before: The email address of the customer
  After:  The email address of the customer

customer_phone [UNCHANGED]
  Before: The phone number of the customer
  After:  The phone number of the customer

customer_address [UNCHANGED]
  Before: The add

## 8. (Optional) Promote to LIVE

In [ ]:
# Uncomment to promote to LIVE:
# bda_client.copy_blueprint_stage(
#     blueprintArn=blueprint_arn,
#     sourceStage='DEVELOPMENT',
#     targetStage='LIVE'
# )
# print(f"Blueprint promoted to LIVE: {blueprint_arn}")

print("Uncomment the code above to promote the optimized blueprint to LIVE stage.")

Blueprint promoted to LIVE: arn:aws:bedrock:us-east-1:208264011282:blueprint/77ff645c8614
Uncomment the code above to promote the optimized blueprint to LIVE stage.


## 9. Clean Up

In [ ]:
# Uncomment to delete the blueprint:
# bda_client.delete_blueprint(blueprintArn=blueprint_arn)
# print(f"Deleted blueprint: {blueprint_arn}")

print(f"To clean up manually:")
print(f"  1. Delete blueprint: {blueprint_arn}")
print(f"  2. Empty and delete S3 bucket: {bucket_name}")

Deleted blueprint: arn:aws:bedrock:us-east-1:208264011282:blueprint/77ff645c8614
To clean up manually:
  1. Delete blueprint: arn:aws:bedrock:us-east-1:208264011282:blueprint/77ff645c8614
  2. Empty and delete S3 bucket: bda-workshop-us-east-1-208264011282
